In [1]:
import pandas as pd
import glob

path = "topGlobal"

files = sorted(glob.glob(f"{path}/global*.csv"))

print("Archivos encontrados:", len(files))

if len(files) == 0:
    raise ValueError("❌ No se encontraron CSV en el path indicado")

dfs = []

for f in files:
    print("Leyendo:", f)
    tmp = pd.read_csv(f)
    dfs.append(tmp)

df = pd.concat(dfs, ignore_index=True)

Archivos encontrados: 10
Leyendo: topGlobal/global2016.csv
Leyendo: topGlobal/global2017.csv
Leyendo: topGlobal/global2018.csv
Leyendo: topGlobal/global2019.csv
Leyendo: topGlobal/global2020.csv
Leyendo: topGlobal/global2021.csv
Leyendo: topGlobal/global2022.csv
Leyendo: topGlobal/global2023.csv
Leyendo: topGlobal/global2024.csv
Leyendo: topGlobal/global2025.csv


In [3]:
df

,rank,uri,artist_names,track_name,source,peak_rank,previous_rank,weeks_on_chart,streams,source_file
0,1,spotify:track:5aAx2yezTd8zXrkmtKl66Z,"The Weeknd, Daft Punk",Starboy,The Weeknd/Lyric,1,-1,1,25286465,regional-global-weekly-2016-12-29.csv
1,2,spotify:track:7BKLCZ1jbUBVqRi2FVlTVw,"The Chainsmokers, Halsey",Closer,Disruptor Records/Columbia,2,-1,1,22047697,regional-global-weekly-2016-12-29.csv
2,3,spotify:track:5knuzwU65gJK7IF5yJsuaW,"Clean Bandit, Anne-Marie, Sean Paul",Rockabye (feat. Sean Paul & Anne-Marie),Atlantic Records UK,3,-1,1,19794482,regional-global-weekly-2016-12-29.csv
3,4,spotify:track:4pdPtRcBmOSQDlJ3Fk945m,"DJ Snake, Justin Bieber",Let Me Love You,DJ Snake Def Jam,4,-1,1,17965723,regional-global-weekly-2016-12-29.csv
4,5,spotify:track:5MFzQMkrl1FOOng9tq6R9r,"Maroon 5, Kendrick Lamar",Don't Wanna Know,Interscope Records*,5,-1,1,16966668,regional-global-weekly-2016-12-29.csv
...,...,...,...,...,...,...,...,...,...,...
93971,196,spotify:track:1k0JAiH11gHL9dc5dfQjQr,ILLIT,NOT CUTE ANYMORE,BELIFT LAB,161,161,4,9425843,regional-global-weekly-2025-12-25.csv
93972,197,spotify:track:1PREzVLuDT6PSE9sej4wnV,"Fuerza Regida, Grupo Frontera",COQUETA,Rancho Humilde/Streetmob Records/Grupo Fronter...,88,180,35,9414932,regional-global-weekly-2025-12-25.csv
93973,198,spotify:track:3nfxV3jfRcRWISwek49LTg,"Michael Bublé, The Puppini Sisters",Jingle Bells (feat. The Puppini Sisters),Reprise,136,-1,7,9380932,regional-global-weekly-2025-12-25.csv
93974,199,spotify:track:4hnzXzE7JiU5QAZFF9OUsI,"John Legend, Stevie Wonder",What Christmas Means to Me (feat. Stevie Wonder),Columbia/Legacy,199,-1,1,9351307,regional-global-weekly-2025-12-25.csv


In [4]:
import pandas as pd

# Assume your dataframe is called df and column is 'artist_names'

# Split artists by comma and explode into rows
artists_series = (
    df["artist_names"]
      .dropna()
      .str.split(",")
      .explode()
      .str.strip()
)

# Drop duplicates and sort (optional but nice)
unique_artists = (
    artists_series
    .drop_duplicates()
    .sort_values()
)

# Convert to DataFrame
artists_df = pd.DataFrame({
    "artist_name": unique_artists
})

# Save to CSV
artists_df.to_csv("unique_artists.csv", index=False)

print(f"Saved {len(artists_df)} unique artists to unique_artists.csv")


Saved 2463 unique artists to unique_artists.csv


In [5]:
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials

CLIENT_ID = "2a1f45380fda4f0ba4661f6e44882f59"
CLIENT_SECRET = "79f5af5a520f4a37bb66b2503e331635"

sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET
    )
)

In [7]:
sp.track("6RUKPb4LETWmmr3iAEQktW")["name"]

'Something Just Like This'

In [8]:
import pandas as pd
import spotipy
from spotipy.oauth2 import SpotifyClientCredentials
from tqdm import tqdm
import time

# ==========
# Auth (mejor con env vars)
# export SPOTIPY_CLIENT_ID="..."
# export SPOTIPY_CLIENT_SECRET="..."
# ==========
sp = spotipy.Spotify(
    auth_manager=SpotifyClientCredentials(
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET
    )
)

def chunks(lst, n):
    for i in range(0, len(lst), n):
        yield lst[i:i+n]

# --- 1) track_id desde uri
df = df.copy()
df["track_id"] = df["uri"].astype(str).str.split(":").str[-1]

track_ids = df["track_id"].dropna().unique().tolist()

# --- 2) bajar metadata de tracks (album_id, album_name, release_date, etc.)
track_rows = []
for batch in tqdm(list(chunks(track_ids, 50)), desc="Tracks"):
    try:
        resp = sp.tracks(batch)  # {'tracks': [...]}
        for t in resp["tracks"]:
            if not t:
                continue
            track_rows.append({
                "track_id": t["id"],
                "track_popularity": t.get("popularity"),
                "explicit": t.get("explicit"),
                "duration_ms": t.get("duration_ms"),
                "album_id": t["album"]["id"] if t.get("album") else None,
                "album_name": t["album"]["name"] if t.get("album") else None,
                "album_release_date": t["album"].get("release_date") if t.get("album") else None,
                "album_type": t["album"].get("album_type") if t.get("album") else None,
                "album_total_tracks": t["album"].get("total_tracks") if t.get("album") else None,
                "artist_ids_api": ",".join([a["id"] for a in t.get("artists", []) if a.get("id")]),
                "artist_names_api": ", ".join([a["name"] for a in t.get("artists", [])]),
            })
        time.sleep(0.1)
    except Exception as e:
        print("[WARN] tracks batch falló:", e)
        continue

tracks_df = pd.DataFrame(track_rows).drop_duplicates("track_id")

# --- 3) bajar metadata de albums (aquí vive el label)
album_ids = tracks_df["album_id"].dropna().unique().tolist()

album_rows = []
for batch in tqdm(list(chunks(album_ids, 20)), desc="Albums (label)"):
    try:
        resp = sp.albums(batch)  # {'albums': [...]}
        for a in resp["albums"]:
            if not a:
                continue
            album_rows.append({
                "album_id": a["id"],
                "label": a.get("label"),                       # ✅ proxy principal
                "copyrights": " | ".join([c.get("text","") for c in a.get("copyrights", [])]),
                "album_popularity": a.get("popularity"),       # a veces viene, a veces no
                "album_genres": ", ".join(a.get("genres", []))
            })
        time.sleep(0.1)
    except Exception as e:
        print("[WARN] albums batch falló:", e)
        continue

albums_df = pd.DataFrame(album_rows).drop_duplicates("album_id")

# --- 4) merge final: df original + track meta + album meta
df_enriched = (
    df
    .merge(tracks_df, on="track_id", how="left")
    .merge(albums_df, on="album_id", how="left")
)

print("Listo. % con label:", df_enriched["label"].notna().mean())

# opcional: guarda
df_enriched.to_csv("topGlobal/df_with_album_label.csv", index=False)

Tracks:  82%|████████▏ | 127/154 [01:08<00:14,  1.86it/s]


KeyboardInterrupt: 